# 부서 연결 Retriever 프로젝트

## 변경 목적

1. Colab GPU 사용 전제
2. `업무담당규정.docx`는 검색용 corpus chunk로 사용하지 않음
3. `업무담당규정.docx`는 1차 도메인 적응 학습용 텍스트로만 사용
4. `dept.xlsx`는 `HNG_BR_NM`, `OFWRK_NM` 컬럼 기준으로 로드
5. `dept.xlsx`는 **부서별 5행 chunk**로 분할하여 최종 검색 corpus로 사용
6. SentenceTransformer 학습을 2회로 분리
   - 1차: `업무담당규정.docx` 기반 도메인 적응
   - 2차: `dept.xlsx` 라벨 기반 부서 연결 학습
7. query 입력 시 `HNG_BR_NM` 상위 5개 출력 후, 다수결 방식으로 최종 1건 출력

> 주의: SentenceTransformer는 LLM처럼 문서를 외워서 답변하는 모델이 아닙니다.  
> 여기서는 `업무담당규정.docx` 문체와 업무 용어에 embedding 공간을 적응시키는 실험 구조입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# 0. Colab GPU 환경 세팅
# ============================================================
# 메뉴에서 먼저 설정하세요.
# 런타임 > 런타임 유형 변경 > 하드웨어 가속기 > GPU

!nvidia-smi

# 최초 1회 설치
!pip install -U sentence-transformers rank_bm25 python-docx openpyxl tqdm scikit-learn datasets


  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.5.1
    Uninstalling sentence-transformers-5.5.1:
      Successfully uninstalled sentence-transformers-5.5.1


In [ ]:
# ============================================================
# 1. 라이브러리 및 기본 설정
# ============================================================

import os
import re
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

import torch
from torch.utils.data import DataLoader

from sentence_transformers import SentenceTransformer, InputExample, losses

# sentence-transformers 일부 버전에서 fit 내부 Dataset NameError가 나는 경우 방지
from datasets import Dataset
import sentence_transformers.sentence_transformer.fit_mixin as fit_mixin
fit_mixin.Dataset = Dataset

# GPU 사용
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

if DEVICE != "cuda":
    raise RuntimeError("GPU가 잡히지 않았습니다. Colab 메뉴에서 런타임 > 런타임 유형 변경 > GPU로 바꾸세요.")

# Colab GPU에서 matmul 성능 개선용
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("DEVICE:", DEVICE)
print("USE_AMP:", USE_AMP)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))


DEVICE: cuda
USE_AMP: True
torch: 2.11.0+cu128
GPU: Tesla T4


/tmp/ipykernel_869/3853369828.py:21: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [ ]:
# ============================================================
# 2. 파일 경로 및 학습 옵션
# ============================================================
# 방법 A: Colab 왼쪽 파일탭에 업무담당규정.docx, dept.xlsx 업로드 후 /content 기준 사용
# 방법 B: Google Drive에 저장 후 PROJECT_DIR만 바꿔서 사용

# Google Drive 사용 시 주석 해제
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/AX')

# 세션 업로드 파일을 사용할 경우
# PROJECT_DIR = Path('/content')

BASE_DOC_PATH = "/content/drive/MyDrive/AX/업무담당규정.docx"
DEPT_XLSX_PATH = "/content/drive/MyDrive/AX/dept.xlsx"

# 기본 SentenceTransformer 모델
EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"

# 2단계 학습 결과 저장 경로
STAGE1_MODEL_DIR = str(PROJECT_DIR / "stage1_domain_retriever")
STAGE2_MODEL_DIR = str(PROJECT_DIR / "stage2_dept_retriever")

# 테스트 목적이면 True
# 이미 학습된 모델을 재사용하려면 False
RUN_STAGE1_DOMAIN_TRAIN = True
RUN_STAGE2_DEPT_TRAIN = True

# 1차 도메인 학습 설정: GPU 기준
STAGE1_EPOCHS = 1
STAGE1_BATCH_SIZE = 32
STAGE1_LR = 2e-5
STAGE1_MAX_TRAIN_TEXTS = 5000   # 전체 문단 사용하려면 None

# 2차 dept 학습 설정: GPU 기준
STAGE2_EPOCHS = 1
STAGE2_BATCH_SIZE = 32
STAGE2_LR = 2e-5

# dept.xlsx chunk 규칙
DEPT_CHUNK_SIZE = 5

# 검색 설정
DENSE_WEIGHT = 0.55
TFIDF_WEIGHT = 0.25
BM25_WEIGHT = 0.20

# 임베딩 생성 배치 크기
ENCODE_BATCH_SIZE = 128

REFERENCE_TOP_ROWS = 5

print("BASE_DOC_PATH:", BASE_DOC_PATH)
print("DEPT_XLSX_PATH:", DEPT_XLSX_PATH)
print("STAGE1_MODEL_DIR:", STAGE1_MODEL_DIR)
print("STAGE2_MODEL_DIR:", STAGE2_MODEL_DIR)


BASE_DOC_PATH: /content/drive/MyDrive/AX/업무담당규정.docx
DEPT_XLSX_PATH: /content/drive/MyDrive/AX/dept.xlsx
STAGE1_MODEL_DIR: /content/drive/MyDrive/AX/stage1_domain_retriever
STAGE2_MODEL_DIR: /content/drive/MyDrive/AX/stage2_dept_retriever


In [ ]:
# ============================================================
# 2-1. 파일 존재 확인
# ============================================================

for p in [BASE_DOC_PATH, DEPT_XLSX_PATH]:
    if not Path(p).exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {p}
Colab 파일탭에 업로드했는지, PROJECT_DIR 경로가 맞는지 확인하세요.")

print("파일 확인 완료")


In [ ]:
# ============================================================
# 3. dept.xlsx 로드 및 검증
# ============================================================
# 필수 컬럼:
# - HNG_BR_NM : 부서명
# - OFWRK_NM  : 업무 설명 / 역할 설명

def clean_text(x) -> str:
    x = "" if pd.isna(x) else str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def load_dept_excel(path: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"dept.xlsx 파일을 찾을 수 없습니다: {path.resolve()}")

    df = pd.read_excel(path)

    required_cols = ["HNG_BR_NM", "OFWRK_NM"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"dept.xlsx에 필요한 컬럼이 없습니다: {missing}\n"
            f"현재 컬럼: {list(df.columns)}"
        )

    df = df[required_cols].copy()
    df["HNG_BR_NM"] = df["HNG_BR_NM"].apply(clean_text)
    df["OFWRK_NM"] = df["OFWRK_NM"].apply(clean_text)

    df = df.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    df = df.dropna(subset=["HNG_BR_NM", "OFWRK_NM"]).reset_index(drop=True)

    # 원본 행번호 추적용
    df["orig_row_id"] = np.arange(len(df))

    return df


dept_df = load_dept_excel(DEPT_XLSX_PATH)

print("dept_df shape:", dept_df.shape)
display(dept_df.head())
print("부서 수:", dept_df["HNG_BR_NM"].nunique())


dept_df shape: (2544, 3)


,HNG_BR_NM,OFWRK_NM,orig_row_id
0,디지털 HR부,"[Cell장 담당업무]디지털/ICT 인사 운용, 채용, 연수 전반사항 [담당] AX...",0
1,디지털 HR부,"디지털/ICT 연수 기획/운영, 신입직원 연수/온보딩",1
2,디지털 HR부,"디지털/ICT 연수 기획/운영, 신입직원 채용/연수/온보딩",2
3,디지털 HR부,"디지털/ICT 직원 인사 [담당] Tech그룹, 정보보호본부, 기관·제휴개발부",3
4,디지털 HR부,"디지털/ICT 채용, 직무 및 CDP, 신입직원 연수/온보딩",4


부서 수: 103


In [ ]:
# ============================================================
# 4. 업무담당규정.docx 로드
# ============================================================
# 이번 버전에서는 업무담당규정.docx를 검색 corpus chunk로 만들지 않습니다.
# 대신 1차 도메인 적응 학습용 문단 데이터로만 사용합니다.

def read_docx_paragraphs(docx_path: str, min_chars: int = 20) -> list:
    from docx import Document

    path = Path(docx_path)
    if not path.exists():
        raise FileNotFoundError(f"업무담당규정.docx 파일을 찾을 수 없습니다: {path.resolve()}")

    doc = Document(str(path))
    paragraphs = []

    for p in doc.paragraphs:
        t = clean_text(p.text)
        if len(t) >= min_chars:
            paragraphs.append(t)

    # 표 안의 텍스트도 추가
    for table in doc.tables:
        for row in table.rows:
            row_texts = []
            for cell in row.cells:
                t = clean_text(cell.text)
                if t:
                    row_texts.append(t)
            merged = " ".join(row_texts)
            if len(merged) >= min_chars:
                paragraphs.append(merged)

    # 중복 제거, 순서 유지
    seen = set()
    unique_paragraphs = []
    for t in paragraphs:
        if t not in seen:
            unique_paragraphs.append(t)
            seen.add(t)

    return unique_paragraphs


domain_texts = read_docx_paragraphs(BASE_DOC_PATH, min_chars=20)

print("업무담당규정 도메인 학습 문단 수:", len(domain_texts))
print("\n[샘플]")
for t in domain_texts[:5]:
    print("-", t[:200])


업무담당규정 도메인 학습 문단 수: 2114

[샘플]
- 제1조(목적) 이 규정은 본부조직 및 영업조직의 담당업무를 정하는 데 그 목적이 있다. 단, 각 부서는 본 규정의 담당업무에도 불구하고, 은행의 중장기적인 가치 향상을 위한 부서간 협업에 역량과 자원을 우선 배분해야 하며, 별도의 명시 및 외규, 내규의 제약이 없는 한 본 규정의 효력 범위를 국내에 제한하지 않는다.
- 제2조(기안) 이 규정의 제정, 개폐에 기안권자는 종합기획부장으로 한다. 단, 다음과 같은 사항과 관련된 기안권자는 소관부서장으로 하되 종합기획부장의 합의를 거치도록 한다.
- 1. 동일 그룹 또는 본부 내 부서간 업무조정에 따른 개정
- 2. 타 그룹 또는 본부에 영향을 미치지 않는 담당업무의 추가, 신규에 따른 개정
- 3. 은행장이 별도로 결정한 사항에 따른 부수적인 개정


In [ ]:
# ============================================================
# 5. 1차 학습: 업무담당규정.docx 기반 도메인 적응
# ============================================================
# chunk 검색용으로 나누지 않고, 문단 자체를 self-positive pair로 사용합니다.
# 목적:
# - 업무담당규정의 용어, 문체, 부서 업무 표현에 embedding 공간을 적응
#
# 방식:
# - InputExample(texts=[문단, 같은 문단])
# - MultipleNegativesRankingLoss 사용
#
# 주의:
# - 이 방식은 LLM의 "지식 주입"과 다릅니다.
# - SentenceTransformer retriever의 표현공간을 도메인 문장에 맞추는 테스트용 설계입니다.
# - Colab GPU에서는 use_amp=True로 학습 속도를 올립니다.

def prepare_stage1_domain_examples(domain_texts: list, max_train_texts=1000) -> list:
    texts = [clean_text(t) for t in domain_texts if len(clean_text(t)) >= 20]

    # 너무 긴 문단은 학습 안정성을 위해 자름
    texts = [t[:512] for t in texts]

    # 중복 제거
    texts = list(dict.fromkeys(texts))

    if max_train_texts is not None and len(texts) > max_train_texts:
        random.shuffle(texts)
        texts = texts[:max_train_texts]

    examples = [InputExample(texts=[t, t]) for t in texts]
    return examples


def train_stage1_domain_model(
    base_model_name: str,
    domain_texts: list,
    output_dir: str,
    epochs: int = 1,
    batch_size: int = 8,
    lr: float = 2e-5,
    max_train_texts=1000
):
    train_examples = prepare_stage1_domain_examples(
        domain_texts=domain_texts,
        max_train_texts=max_train_texts
    )

    if len(train_examples) == 0:
        raise ValueError("1차 학습용 domain_texts가 비어 있습니다.")

    print("1차 학습 예제 수:", len(train_examples))

    model = SentenceTransformer(base_model_name, device=DEVICE)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    warmup_steps = max(1, int(len(train_loader) * 0.1))

    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": lr},
        show_progress_bar=True,
        use_amp=USE_AMP
    )

    model.save(output_dir)
    print("1차 도메인 모델 저장 완료:", output_dir)
    return model


if RUN_STAGE1_DOMAIN_TRAIN:
    stage1_model = train_stage1_domain_model(
        base_model_name=EMBEDDING_MODEL_NAME,
        domain_texts=domain_texts,
        output_dir=STAGE1_MODEL_DIR,
        epochs=STAGE1_EPOCHS,
        batch_size=STAGE1_BATCH_SIZE,
        lr=STAGE1_LR,
        max_train_texts=STAGE1_MAX_TRAIN_TEXTS
    )
    STAGE1_BASE_FOR_STAGE2 = STAGE1_MODEL_DIR
else:
    if Path(STAGE1_MODEL_DIR).exists():
        print("기존 1차 모델 사용:", STAGE1_MODEL_DIR)
        STAGE1_BASE_FOR_STAGE2 = STAGE1_MODEL_DIR
    else:
        print("1차 학습 생략. 기본 모델 사용:", EMBEDDING_MODEL_NAME)
        STAGE1_BASE_FOR_STAGE2 = EMBEDDING_MODEL_NAME


1차 학습 예제 수: 2114


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

1차 도메인 모델 저장 완료: /content/drive/MyDrive/AX/stage1_domain_retriever


In [ ]:
# ============================================================
# 6. dept.xlsx 5행 chunk 생성
# ============================================================
# 핵심 변경:
# - 전체 엑셀을 단순 5행씩 자르면 서로 다른 부서가 섞일 수 있습니다.
# - 따라서 "부서별로 묶은 뒤, 각 부서 내부에서 5행씩 chunk"로 만듭니다.
# - 이렇게 해야 각 chunk가 단일 HNG_BR_NM을 갖고, 다수결 투표가 깔끔하게 동작합니다.

def build_dept_chunks_by_department(dept_df: pd.DataFrame, chunk_size: int = 5) -> pd.DataFrame:
    rows = []
    chunk_no = 0

    for dept, g in dept_df.groupby("HNG_BR_NM", sort=False):
        g = g.sort_values("orig_row_id").reset_index(drop=True)

        for start in range(0, len(g), chunk_size):
            part = g.iloc[start:start + chunk_size].copy()
            works = part["OFWRK_NM"].astype(str).tolist()

            text = (
                f"[부서명] {dept}\n"
                f"[담당업무]\n" +
                "\n".join([f"- {w}" for w in works])
            )

            rows.append({
                "source": "dept.xlsx",
                "chunk_id": f"dept_chunk_{chunk_no}",
                "HNG_BR_NM": dept,
                "text": text,
                "work_items": works,
                "n_rows": len(part),
                "orig_row_start": int(part["orig_row_id"].min()),
                "orig_row_end": int(part["orig_row_id"].max())
            })

            chunk_no += 1

    return pd.DataFrame(rows)


dept_corpus = build_dept_chunks_by_department(dept_df, chunk_size=DEPT_CHUNK_SIZE)

# 이번 구조에서 최종 검색 corpus는 dept.xlsx chunk만 사용합니다.
# 업무담당규정.docx는 1차 학습에만 사용하고, 검색 대상에서는 제외합니다.
corpus_df = dept_corpus[[
    "source", "chunk_id", "HNG_BR_NM", "text", "work_items", "n_rows", "orig_row_start", "orig_row_end"
]].copy()

print("dept chunk corpus shape:", corpus_df.shape)
print("부서 수:", corpus_df["HNG_BR_NM"].nunique())
display(corpus_df.head(10))


dept chunk corpus shape: (559, 8)
부서 수: 103


,source,chunk_id,HNG_BR_NM,text,work_items,n_rows,orig_row_start,orig_row_end
0,dept.xlsx,dept_chunk_0,디지털 HR부,[부서명] 디지털 HR부\n[담당업무]\n- [Cell장 담당업무]디지털/ICT 인...,"[[Cell장 담당업무]디지털/ICT 인사 운용, 채용, 연수 전반사항 [담당] A...",5,0,4
1,dept.xlsx,dept_chunk_1,디지털 HR부,"[부서명] 디지털 HR부\n[담당업무]\n- 디지털/ICT 채용, 직무 및 CDP,...","[디지털/ICT 채용, 직무 및 CDP, 인사제도 운영(부), 민원소통담당자(부),...",2,5,6
2,dept.xlsx,dept_chunk_2,기관영업2부,[부서명] 기관영업2부\n[담당업무]\n- ○ 공공기관 주거래협약 관리 ○ 공공기관...,"[○ 공공기관 주거래협약 관리 ○ 공공기관 계약서 관리 담당자, ○ 공공기관(자금)...",5,7,11
3,dept.xlsx,dept_chunk_3,기관영업2부,[부서명] 기관영업2부\n[담당업무]\n- ○ 대학교/병원 신규유치 및 고객관리 ○...,[○ 대학교/병원 신규유치 및 고객관리 ○ 대학교/병원 만기/재협약 관리 ○ 대학교...,5,12,16
4,dept.xlsx,dept_chunk_4,기관영업2부,[부서명] 기관영업2부\n[담당업무]\n- ○ 대학교/병원 신규유치 및 고객관리 ○...,[○ 대학교/병원 신규유치 및 고객관리 ○ 대학교/병원 만기/재협약 관리 ○ 대학교...,5,17,21
5,dept.xlsx,dept_chunk_5,기관영업1부,[부서명] 기관영업1부\n[담당업무]\n- - OCR/비OCR 집계대사 및 물류발송...,"[- OCR/비OCR 집계대사 및 물류발송, AI_OCR 수기고지서 데이터화 작업,...",5,22,26
6,dept.xlsx,dept_chunk_6,기관영업1부,[부서명] 기관영업1부\n[담당업무]\n- ○ 나라사랑카드 사업 관리 ○ 국군 연계...,"[○ 나라사랑카드 사업 관리 ○ 국군 연계 마케팅(정), ○ 사업운영(정) ○ 나라...",5,27,31
7,dept.xlsx,dept_chunk_7,기관영업1부,[부서명] 기관영업1부\n[담당업무]\n- ○ 사업이행 (정) ○ 병무청 발급소 운...,"[○ 사업이행 (정) ○ 병무청 발급소 운영 ○ 유관기관 소통(국방부, 병무청, 군...",5,32,36
8,dept.xlsx,dept_chunk_8,기관영업1부,[부서명] 기관영업1부\n[담당업무]\n- 군 현장영업\n- 사업계획(정) 재무 및...,"[군 현장영업, 사업계획(정) 재무 및 예산관리(정) KPI 관리(정) 부서 수신문...",5,37,41
9,dept.xlsx,dept_chunk_9,기관영업1부,"[부서명] 기관영업1부\n[담당업무]\n- 시도금고운영CELL 총괄, 세입(시스템)...","[시도금고운영CELL 총괄, 세입(시스템) 지자체 금고 시스템 관리_시스템 운영 이...",5,42,46


In [ ]:
# ============================================================
# 7. 2차 학습: dept.xlsx 라벨 기반 부서 연결 학습
# ============================================================
# 학습 데이터:
# - query 역할: OFWRK_NM 개별 업무 문장
# - positive context 역할: 해당 업무가 들어있는 5행 chunk
#
# 즉, 고객 문의/업무 표현이 들어왔을 때 해당 부서 chunk 쪽으로 embedding이 붙도록 학습합니다.

def prepare_stage2_dept_examples(corpus_df: pd.DataFrame) -> list:
    examples = []

    for _, row in corpus_df.iterrows():
        dept = str(row["HNG_BR_NM"])
        chunk_text = str(row["text"])
        work_items = row["work_items"]

        # 1) 개별 업무 문장 -> 해당 5행 chunk
        for w in work_items:
            w = clean_text(w)
            if len(w) >= 3:
                examples.append(InputExample(texts=[w, chunk_text]))

                # 부서명이 포함된 표현도 추가
                examples.append(InputExample(texts=[f"{dept} {w}", chunk_text]))

        # 2) 부서명 자체 -> 해당 chunk
        examples.append(InputExample(texts=[dept, chunk_text]))

    random.shuffle(examples)
    return examples


def train_stage2_dept_model(
    stage1_or_base_model_name: str,
    corpus_df: pd.DataFrame,
    output_dir: str,
    epochs: int = 1,
    batch_size: int = 8,
    lr: float = 2e-5
):
    train_examples = prepare_stage2_dept_examples(corpus_df)

    if len(train_examples) == 0:
        raise ValueError("2차 학습용 dept 예제가 비어 있습니다.")

    print("2차 학습 예제 수:", len(train_examples))

    model = SentenceTransformer(stage1_or_base_model_name, device=DEVICE)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    warmup_steps = max(1, int(len(train_loader) * 0.1))

    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": lr},
        show_progress_bar=True,
        use_amp=USE_AMP
    )

    model.save(output_dir)
    print("2차 dept 모델 저장 완료:", output_dir)
    return model


if RUN_STAGE2_DEPT_TRAIN:
    embedder = train_stage2_dept_model(
        stage1_or_base_model_name=STAGE1_BASE_FOR_STAGE2,
        corpus_df=corpus_df,
        output_dir=STAGE2_MODEL_DIR,
        epochs=STAGE2_EPOCHS,
        batch_size=STAGE2_BATCH_SIZE,
        lr=STAGE2_LR
    )
else:
    if Path(STAGE2_MODEL_DIR).exists():
        print("기존 2차 모델 사용:", STAGE2_MODEL_DIR)
        embedder = SentenceTransformer(STAGE2_MODEL_DIR, device=DEVICE)
    else:
        print("2차 학습 생략. 1차 또는 기본 모델 사용:", STAGE1_BASE_FOR_STAGE2)
        embedder = SentenceTransformer(STAGE1_BASE_FOR_STAGE2, device=DEVICE)

print("최종 embedder 준비 완료")


2차 학습 예제 수: 5639


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2차 dept 모델 저장 완료: /content/drive/MyDrive/AX/stage2_dept_retriever
최종 embedder 준비 완료


In [ ]:
# ============================================================
# 8. 임베딩 / TF-IDF / BM25 인덱스 생성
# ============================================================

def simple_tokenize(text: str) -> list:
    return re.findall(r"[가-힣A-Za-z0-9]+", str(text).lower())


def normalize_matrix(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)


def top_indices(scores: np.ndarray, k: int) -> np.ndarray:
    k = min(k, len(scores))
    return np.argsort(scores)[-k:][::-1]


corpus_texts = corpus_df["text"].astype(str).tolist()

# 1) Dense embedding
corpus_embs = embedder.encode(
    corpus_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=ENCODE_BATCH_SIZE
)
corpus_embs = normalize_matrix(corpus_embs)

# 2) TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_texts)

# 3) BM25
tokenized_corpus = [simple_tokenize(t) for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("Dense embeddings:", corpus_embs.shape)
print("TF-IDF matrix:", tfidf_matrix.shape)
print("BM25 corpus:", len(tokenized_corpus))


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Dense embeddings: (559, 768)
TF-IDF matrix: (559, 100834)
BM25 corpus: 559


In [ ]:
# ============================================================
# 9. Query → 검색결과 상위 5개 행만 참조 + HNG_BR_NM 다수결 최종 1건
# ============================================================

def minmax_norm(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return x
    return (x - x.min()) / (np.ptp(x) + 1e-12)


def retrieve_scored_rows(
    query: str,
    candidate_k: int = 80,
    return_top_rows: int = REFERENCE_TOP_ROWS,
    dense_weight: float = DENSE_WEIGHT,
    tfidf_weight: float = TFIDF_WEIGHT,
    bm25_weight: float = BM25_WEIGHT
) -> pd.DataFrame:
    """
    dense / TF-IDF / BM25를 각각 계산한 뒤,
    각 방식의 상위 후보를 union하여 최종 점수를 계산합니다.

    최종 반환은 정렬된 검색결과 중 상위 return_top_rows개 행만 반환합니다.
    즉, 이후 다수결/최종 판단은 이 상위 5개 행만 참조합니다.
    """
    query = clean_text(query)
    if not query:
        raise ValueError("query가 비어 있습니다.")

    candidate_k = min(candidate_k, len(corpus_df))
    return_top_rows = min(return_top_rows, len(corpus_df))

    # 1) Dense cosine similarity
    q_emb = embedder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-12)
    dense_scores_all = corpus_embs @ q_emb[0]

    # 2) TF-IDF
    q_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores_all = (tfidf_matrix @ q_tfidf.T).toarray().ravel()

    # 3) BM25
    bm25_scores_all = np.asarray(bm25.get_scores(simple_tokenize(query)))

    # 방식별 후보 union
    dense_idx = top_indices(dense_scores_all, candidate_k)
    tfidf_idx = top_indices(tfidf_scores_all, candidate_k)
    bm25_idx = top_indices(bm25_scores_all, candidate_k)

    candidate_idx = np.unique(np.concatenate([dense_idx, tfidf_idx, bm25_idx]))

    d = dense_scores_all[candidate_idx]
    t = tfidf_scores_all[candidate_idx]
    b = bm25_scores_all[candidate_idx]

    final_scores = (
        dense_weight * minmax_norm(d) +
        tfidf_weight * minmax_norm(t) +
        bm25_weight * minmax_norm(b)
    )

    result = corpus_df.iloc[candidate_idx].copy()
    result["dense_score"] = d
    result["tfidf_score"] = t
    result["bm25_score"] = b
    result["score"] = final_scores

    result = result.sort_values("score", ascending=False).reset_index(drop=True)
    result["rank"] = np.arange(1, len(result) + 1)

    # 핵심 수정: 검색결과 상위 5개 행만 반환
    result = result.head(return_top_rows).copy()
    result["rank"] = np.arange(1, len(result) + 1)

    return result


def predict_department(
    query: str,
    top_n_departments: int = 5,
    candidate_k: int = 80,
    reference_top_rows: int = REFERENCE_TOP_ROWS
):
    """
    1) query와 유사한 chunk 후보 검색
    2) 검색결과 상위 reference_top_rows개 행만 참조
    3) HNG_BR_NM 기준 다수결
    4) 동률이면 score_sum, score_max, rank_weight_sum 순으로 정렬

    기본값:
    - reference_top_rows=5
    """
    scored_rows = retrieve_scored_rows(
        query,
        candidate_k=candidate_k,
        return_top_rows=reference_top_rows
    )

    # 핵심 수정: 최종 판단은 검색결과 상위 5개 행만 참조
    vote_pool = scored_rows.copy()

    # rank가 높을수록 가중치를 크게 주는 보조 지표
    vote_pool["rank_weight"] = 1 / vote_pool["rank"]

    summary = (
        vote_pool
        .groupby("HNG_BR_NM")
        .agg(
            vote_count=("HNG_BR_NM", "size"),
            score_sum=("score", "sum"),
            score_max=("score", "max"),
            rank_weight_sum=("rank_weight", "sum"),
            best_rank=("rank", "min"),
            best_source=("source", "first"),
            best_chunk_id=("chunk_id", "first"),
            best_text=("text", "first")
        )
        .reset_index()
    )

    summary = summary.sort_values(
        ["vote_count", "score_sum", "score_max", "rank_weight_sum", "best_rank"],
        ascending=[False, False, False, False, True]
    ).reset_index(drop=True)

    top5 = summary.head(top_n_departments).copy()
    final_department = top5.iloc[0]["HNG_BR_NM"] if len(top5) else None

    return {
        "query": query,
        "final_department": final_department,
        "top5_departments": top5,
        "scored_rows": scored_rows,
        "vote_pool": vote_pool
    }


def print_prediction(result: dict):
    print("[입력 Query]")
    print(result["query"])

    print("\n[참조한 검색결과 상위 5개 행]")
    scored_rows = result["scored_rows"]
    if len(scored_rows) == 0:
        print("검색 결과 없음")
        return

    for _, row in scored_rows.iterrows():
        print(
            f"{int(row['rank'])}. {row['HNG_BR_NM']} "
            f"| score={row['score']:.4f} "
            f"| chunk_id={row['chunk_id']}"
        )

    print("\n[HNG_BR_NM 상위 5개 - 상위 5개 행 다수결 기준]")
    top5 = result["top5_departments"]
    for i, row in top5.iterrows():
        print(
            f"{i+1}. {row['HNG_BR_NM']} "
            f"| votes={int(row['vote_count'])} "
            f"| score_sum={row['score_sum']:.4f} "
            f"| score_max={row['score_max']:.4f}"
        )

    print("\n[최종 출력 1건]")
    print(result["final_department"])


In [ ]:
# ============================================================
# 10. 단건 테스트
# ============================================================

with open("/content/drive/MyDrive/AX/query1.txt", "r", encoding="utf-8") as f:
    query1 = f.read()

result = predict_department(
    query=query1,
    top_n_departments=5,
    candidate_k=80,
    reference_top_rows=REFERENCE_TOP_ROWS
)

print_prediction(result)

display(result["scored_rows"].head(5)[[
    "rank", "HNG_BR_NM", "source", "chunk_id", "score",
    "dense_score", "tfidf_score", "bm25_score", "text"
]])


[입력 Query]
“마이데이터 활용 금리인하요구 서비스” 등 혁신금융서비스 3건 신규 지정 의결 -금번 신규 지정으로 혁신금융서비스 누적 지정건수는 총 1,075건
2026-06-17 조회수 : 532
담당부서디지털금융총괄과 담당자송현지 서기관 연락처02-2100-2841
담당부서디지털금융총괄과 담당자이혜인 사무관 연락처02-2100-2872


“마이데이터 활용 금리인하요구 서비스” 등

 

혁신금융서비스 3건 신규 지정 의결



-금번 신규 지정으로 혁신금융서비스 누적 지정건수는 총 1,075건





금융위원회(위원장 이억원)는 6월 17일 정례회의를 통해 3건의 혁신금융서비스를 신규로 지정하였다. 이로써 현재까지 총 1,075건의 서비스가 혁신금융서비스로 지정되어 시장에서 테스트할 수 있게 되었다.



혁신금융서비스 지정 주요 내용은 다음과 같다.(“금융위 의결 결과 세부내용” : 참고)



금융위원회는 농업협동조합중앙회의 ‘마이데이터 활용 금리인하요구 서비스’를 혁신금융서비스로 신규 지정하였다. 이에 따라 농협협동조합중앙회가 제공하는 마이데이터서비스를 활용하여 적시에 금리인하요구권을 행사할 수 있게 되는 바 대출이자 부담 절감, 이용자의 시간·노력 감소 등 금융소비자 편익이 증가할 것으로 기대된다.



 또한, SK플래닛 및 전북은행의 ‘OK 캐시백 제휴통장 서비스’도 혁신금융서비스로 신규 지정하였다. SK플래닛이 전북은행 제휴 통장 개설을 중개하고 제휴계좌 상품을 광고할 수 있도록 규제 특례를 부여하였다. 이를 통해 OK캐쉬백 앱 이용자는 예금자보호가 적용되는 전북은행 제휴계좌에 SK플래닛이 발행한 선불전자지급수단을 보관 할 수 있게 되고 예금이자 및 제휴혜택 등을 제공받는 등 소비자 효용이 제고된다.



더불어, KB자산운용의 ‘개인형 퇴직연금 로보어드바이저 일임서비스’가 혁신금융서비스로 신규 지정되었다. 로보어드바이저가 알고리즘 등을 통해 투자자 성향별 맞춤형 포트폴리오를 자동 생성하고 개인형퇴직연금(IRP) 적립금에 대한 일임운용

,rank,HNG_BR_NM,source,chunk_id,score,dense_score,tfidf_score,bm25_score,text
0,1,Mydata Unit,dept.xlsx,dept_chunk_119,0.747572,0.298099,0.141002,50.953624,[부서명] Mydata Unit\n[담당업무]\n- ▷ 마이데이터 기반 플랫폼 금융...
1,2,Mydata Unit,dept.xlsx,dept_chunk_121,0.700860,0.256725,0.124768,59.653277,[부서명] Mydata Unit\n[담당업무]\n- ▷ 자산연결 증대 마케팅 ▷ 부...
2,3,업무혁신부,dept.xlsx,dept_chunk_459,0.665505,0.501126,0.046704,12.520449,[부서명] 업무혁신부\n[담당업무]\n- - 정보제공요구서 접수 - 정보제공 요구 ...
3,4,고객플랫폼부,dept.xlsx,dept_chunk_111,0.656327,0.345383,0.088977,37.268886,[부서명] 고객플랫폼부\n[담당업무]\n- ▶ SOL뱅크 MAU/상품/신규 마케팅 ...
4,5,고객상담센터,dept.xlsx,dept_chunk_517,0.633515,0.323306,0.073732,46.215927,[부서명] 고객상담센터\n[담당업무]\n- - 지능형 상담서비스 배포 및 운영관련 ...


In [ ]:
with open("/content/drive/MyDrive/AX/query2.txt", "r", encoding="utf-8") as f:
    query2 = f.read()

result = predict_department(
    query=query2,
    top_n_departments=5,
    candidate_k=80,
    reference_top_rows=REFERENCE_TOP_ROWS
)

print_prediction(result)

display(result["scored_rows"].head(5)[[
    "rank", "HNG_BR_NM", "source", "chunk_id", "score",
    "dense_score", "tfidf_score", "bm25_score", "text"
]])


[입력 Query]
연체 채무자의 부담을 가중시키는 금융회사의 채권매각 관행을 바로잡겠습니다
2026년 6월 17일
·
서민금융과
본문


< ｢개인 연체채권 관리 강화방안｣ 후속조치 ➋ >

 

 연체 채무자의 부담을 가중시키는

금융회사의 채권매각 관행을 바로잡겠습니다



✓대출을 일으킨 원채권 금융회사가 연체 발생시 해당 채권을 매각하여 손쉽게 고객보호 책임으로부터 절연되면서 채권을 회수하는 관행을 개선

 

 ➊ 원채권 금융회사에 채권매각 이후 양수인의 불법행위에 대한 점검·보고의무 부여

 

 ➋ 채권매각시 매각계약서에 재매각시 승계되는 채무자 보호 조건 등 재매각 관련 사항을 포함하도록 의무화

 

 ☞ 관련 규정(｢채권추심 및 대출채권 매각 가이드라인｣) 개정을 ‘26.7월 중 완료하여 개정완료 즉시 시행

 

✓연체채권 관리 공시시스템 마련, 금융회사 자체 채무조정 강화 등 ｢개인 연체채권 관리 강화방안｣의 다른 조치 필요사항들도 조속히 추진할 계획

 

1



추진배경

 

  금융위원회는 ｢포용적 금융 대전환｣ 제2차 회의(‘26.2.26일)에서 ｢연체자 보호와 신속한 재기 지원을 위한 개인 연체채권 관리 강화방안(붙임 참조)｣을 발표한 바 있다. 오늘 사전 예고한 ｢채권추심 및 대출채권 매각 가이드라인(이하 채권 추심·매각 가이드라인)｣ 개정안은 동 방안에 따른 후속조치로 마련되었다.

 

2



｢채권 추심·매각 가이드라인｣ 개정안 주요내용

 

  이번 개정안이 시행되면 대출을 일으킨 원채권 금융회사는 채권매각 이후에도 채무자 보호책임을 부담하게 된다.

  현행 규율체계 하에서 금융회사가 연체채권을 매각하지 않고 직접 보유하면서 추심하는 경우에는 ｢개인채무자보호법｣(‘24.10월 시행)에 따라 엄격한 추심행위 규제*를 적용받는다. 추심업무를 외부에 위탁하는 경우에도 수탁 채권추심회사가 개인 채무자에게 손해를 발생시킨 경우에 그 채권추심회사와 연대하여 손해를 배상할 책임을 지는 등 강한 관리·감독책임**을 부담하게 

,rank,HNG_BR_NM,source,chunk_id,score,dense_score,tfidf_score,bm25_score,text
0,1,여신관리부,dept.xlsx,dept_chunk_357,0.951218,0.488731,0.147443,85.440121,[부서명] 여신관리부\n[담당업무]\n- 여신 시효관리 업무(B/S채권 포함)(정)...
1,2,여신관리부,dept.xlsx,dept_chunk_359,0.853654,0.493629,0.096036,80.766048,[부서명] 여신관리부\n[담당업무]\n- - 단기 연체 관리\n- 정리여신관리팀 및...
2,3,여신관리부,dept.xlsx,dept_chunk_350,0.818429,0.425617,0.106969,87.505068,[부서명] 여신관리부\n[담당업무]\n- - (정) 개인회생 법원문서 접수 및 원장...
3,4,여신관리부,dept.xlsx,dept_chunk_356,0.784605,0.446927,0.110508,55.588944,[부서명] 여신관리부\n[담당업무]\n- - (정) 일반회생 업무 전반 - (정) ...
4,5,여신관리부,dept.xlsx,dept_chunk_349,0.764619,0.366639,0.092737,103.629600,[부서명] 여신관리부\n[담당업무]\n- - 개인회생 채무자 전산 원장 정비 및 등...


In [ ]:
with open("/content/drive/MyDrive/AX/query3.txt", "r", encoding="utf-8") as f:
    query3 = f.read()

result = predict_department(
    query=query3,
    top_n_departments=5,
    candidate_k=80,
    reference_top_rows=REFERENCE_TOP_ROWS
)

print_prediction(result)

display(result["scored_rows"].head(5)[[
    "rank", "HNG_BR_NM", "source", "chunk_id", "score",
    "dense_score", "tfidf_score", "bm25_score", "text"
]])

[입력 Query]
잘『알고투자』하는 금융교육으로 국민의 안정적 자산형성을 지원하겠습니다. - 금융위원회 부위원장, ‘26년 제1차 금융교육협의회 개최
2026년 6월 16일
·
금융소비자정책과
본문


잘『알고투자』하는 금융교육으로

국민의 안정적 자산형성을 지원하겠습니다.

 

- 금융위원회 부위원장, ‘26년 제1차 금융교육협의회 개최 -



 

◈ 자본시장에 대한 국민적 관심과 개인투자자 참여 확대에 맞춰 ‘자본시장·금융투자 분야 금융교육 강화방안’ 마련

 

◈ 잘 ‘알고투자’ 4대 추진전략을 통해 금융투자교육을 「알기 쉽게 배우고, 고르게 확산하고, 투자판단 역량과 자기보호 역량을 함께 제고」할 수 있도록 추진

 

【관련 국정과제】 82-3번, 청년 등 전국민 경제·금융교육 강화

 

  금융위원회 권대영 부위원장은 6.16일(화) 관계부처 위원 및 민간전문가 등과 함께 「2026년 제1차 금융교육협의회」를 개최하여 ｢자본시장·금융투자 분야 금융교육 강화방안」등에 대해 논의하였다.







< 회의 개요 >







 

◇ 일시/장소 : 2026.6.16.(화) 15:00, 정부서울청사 16층 대회의실

 

◇ 참석기관

 

 - 금융위·재경부·교육부·국방부·행안부·성평등부·공정위 및 금감원

 - 금융공공기관, 교육기관, 협회, 소비자단체, 연구단체 등 19개 관계기관*

 

   * 예금보험공사, 서민금융진흥원, 신용회복위원회, 주택금융공사, 신용보증기금, 한국소비자원, 청소년금융교육협의회, 시니어금융교육협의회, 금융소비자보호재단, 한국YWCA연합회, 한국금융교육학회, 금융연구원, 금융협회(7개)

 

◇ 논의 안건 : ❶ 자본시장·금융투자 분야 금융교육 강화방안
❷ 2025년 금융교육 실적 및 2026년 추진방안
❸ 고령층 대상 노후자산관리 금융교육 추진방안

 

  권대영 부위원장은 모두발언을 통해 “올해는 2021년 3월 금융소비자보호법 시행에 따라 금융교육협의회가 법정기구로 출범한 지 5년째 되는 해”라고 언

,rank,HNG_BR_NM,source,chunk_id,score,dense_score,tfidf_score,bm25_score,text
0,1,사회공헌부,dept.xlsx,dept_chunk_501,0.991879,0.451441,0.415370,283.646914,[부서명] 사회공헌부\n[담당업무]\n- - 금융교육 전략수립 및 실적 관리(정) ...
1,2,사회공헌부,dept.xlsx,dept_chunk_502,0.730661,0.232893,0.374104,265.209393,[부서명] 사회공헌부\n[담당업무]\n- - 연간 사회공헌 사업계획(정) - 지정기...
2,3,투자솔루션부,dept.xlsx,dept_chunk_84,0.633861,0.377982,0.154958,109.073175,[부서명] 투자솔루션부\n[담당업무]\n- ○ 투자전략 및 상품전략 영업점 전파 ○...
3,4,소비자보호부,dept.xlsx,dept_chunk_180,0.620807,0.432801,0.117720,43.731283,[부서명] 소비자보호부\n[담당업무]\n- 금융감독원 보고서 담당자(정)\n- 금융...
4,5,소비자보호부,dept.xlsx,dept_chunk_178,0.620231,0.459405,0.065384,50.049286,[부서명] 소비자보호부\n[담당업무]\n- - 전기통신 금융사기 예방업무 추진 - ...


In [ ]:
# # ============================================================
# # 11. 여러 query 배치 예측
# # ============================================================

# def batch_predict(
#     queries: list,
#     top_n_departments: int = 5,
#     candidate_k: int = 80,
#     vote_pool_size: int = 20
# ) -> pd.DataFrame:
#     rows = []

#     for q in tqdm(queries, desc="predict"):
#         r = predict_department(
#             query=q,
#             top_n_departments=top_n_departments,
#             candidate_k=candidate_k,
#             vote_pool_size=vote_pool_size
#         )

#         top5 = r["top5_departments"]["HNG_BR_NM"].tolist()

#         row = {
#             "query": q,
#             "final_department": r["final_department"]
#         }

#         for i in range(top_n_departments):
#             row[f"top{i+1}"] = top5[i] if i < len(top5) else None

#         rows.append(row)

#     return pd.DataFrame(rows)


# sample_queries = [
#     "신탁형 ISA 상품 구성과 운용 관련해서 상담 가능한 부서를 알려주세요.",
#     "땡겨요 앱에 제가 운영하는 식당을 입점시키려면 어느 부서로 연결해야 하나요?",
#     "투자상품 원금손실 민원이 있어 판매 과정 적정성 검토 담당 부서를 찾고 있습니다.",
# ]

# batch_result = batch_predict(sample_queries)
# display(batch_result)


In [ ]:
# # ============================================================
# # 12. 평가 데이터가 있을 경우 정확도 계산
# # ============================================================
# # 평가 파일 컬럼 예시:
# # - 문장
# # - 유관부서

# TEST_XLSX_PATH = "./test_set.xlsx"

# def evaluate_if_exists(test_path: str):
#     if not Path(test_path).exists():
#         print(f"평가 파일 없음: {test_path}")
#         return None

#     test_df = pd.read_excel(test_path)

#     required = ["문장", "유관부서"]
#     missing = [c for c in required if c not in test_df.columns]
#     if missing:
#         raise ValueError(f"평가 파일에 필요한 컬럼이 없습니다: {missing}")

#     pred_df = batch_predict(test_df["문장"].astype(str).tolist())

#     out = pd.concat(
#         [test_df.reset_index(drop=True), pred_df.drop(columns=["query"])],
#         axis=1
#     )

#     out["correct_top1"] = out["final_department"].astype(str) == out["유관부서"].astype(str)

#     top_cols = ["top1", "top2", "top3", "top4", "top5"]
#     out["correct_top5"] = out.apply(
#         lambda r: str(r["유관부서"]) in [str(r[c]) for c in top_cols],
#         axis=1
#     )

#     print("Top-1 accuracy:", out["correct_top1"].mean())
#     print("Top-5 accuracy:", out["correct_top5"].mean())

#     return out

# # eval_result = evaluate_if_exists(TEST_XLSX_PATH)
# # display(eval_result.head())


## 실행 순서 요약

1. Colab 메뉴에서 `런타임 > 런타임 유형 변경 > GPU`를 선택한다.
2. `업무담당규정.docx`, `dept.xlsx`를 Colab `/content`에 업로드한다.
   - Drive 사용 시 `drive.mount()` 주석을 풀고 `PROJECT_DIR`를 Drive 경로로 바꾼다.
3. 위에서부터 순서대로 실행한다.
4. GPU 메모리가 부족하면 아래 값을 줄인다.
   - `STAGE1_BATCH_SIZE`
   - `STAGE2_BATCH_SIZE`
   - `ENCODE_BATCH_SIZE`
   - `STAGE1_MAX_TRAIN_TEXTS`
5. 한 번 학습 후 저장 모델을 재사용하려면 다음 실행부터 아래처럼 바꾼다.

```python
RUN_STAGE1_DOMAIN_TRAIN = False
RUN_STAGE2_DEPT_TRAIN = False
```

## 핵심 구조

- `업무담당규정.docx`: 검색 chunk로 쓰지 않고 1차 도메인 적응 학습에만 사용
- `dept.xlsx`: 부서별 5행 chunk로 분할 후 최종 검색 corpus로 사용
- 최종 예측: Dense + TF-IDF + BM25 점수 결합 → 상위 row 후보 → `HNG_BR_NM` 다수결
